# Tokenization and Lemmatization

This notebook performs tokenization, stopword removal, and lemmatization on cleaned text.

In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import WordNetLemmatizer
import nltk
import pandas as pd
nltk.data.path.append('/Users/wxs/nltk_data')

## Load cleaned data

In [4]:
from langdetect import detect
df = pd.read_csv('/Users/wxs/Downloads/customer_support_tickets_cleaned.csv')
df['clean_body'] = df['clean_body'].fillna('')
print(f'Loaded {len(df)} rows')
df = df[df["language"] == "en"].copy()
print(f'language filter: {len(df)} rows')
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

df["is_en_text"] = df["body"].apply(is_english)
df = df[df["is_en_text"]].copy()
df.drop(columns=["is_en_text"], inplace=True)
print(f'langdetect: {len(df)} rows')

Loaded 28261 rows
language filter: 28261 rows
langdetect: 28149 rows


## Tokenization

In [5]:
stop_words = set(ENGLISH_STOP_WORDS)

custom_stopwords = {
    'dear', 'customer', 'support', 'team',
    'hello', 'hi', 'thanks', 'thank',
    'regards', 'please', 'kindly',
    'hope', 'message'
}

def tokenize_text(text):
    if text is None:
        return []

    tokens = text.split()

    tokens = [
        word for word in tokens
        if word not in stop_words
        and word not in custom_stopwords
        and len(word) > 2
    ]

    tokens = list(dict.fromkeys(tokens))

    return tokens

df['tokens'] = df['clean_body'].apply(tokenize_text)
df[['clean_body', 'tokens']].head()

,clean_body,tokens
0,i am writing to report a significant problem w...,"[writing, report, significant, problem, centra..."
1,i hope this message reaches you well i am reac...,"[reaches, reaching, request, detailed, informa..."
2,i hope this message finds you well i am reachi...,"[finds, reaching, request, clarification, bill..."
3,i hope this message reaches you well i am reac...,"[reaches, reaching, ask, compatibility, produc..."
4,dear customer support i hope this message reac...,"[reaches, good, health, eager, learn, features..."


## Lemmatization

In [6]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    result = []
    for word in tokens:
        word = lemmatizer.lemmatize(word, 'v')
        word = lemmatizer.lemmatize(word, 'n')
        result.append(word)

    result = list(dict.fromkeys(result))

    return result

df['processed_tokens'] = df['tokens'].apply(lemmatize_tokens)
df['processed_text'] = df['processed_tokens'].apply(lambda x: ' '.join(x))

df = df[df['processed_text'].str.strip() != ''].reset_index(drop=True)

df[['tokens', 'processed_tokens', 'processed_text']].head()

,tokens,processed_tokens,processed_text
0,"[writing, report, significant, problem, centra...","[write, report, significant, problem, centrali...",write report significant problem centralize ac...
1,"[reaches, reaching, request, detailed, informa...","[reach, request, detail, information, capabili...",reach request detail information capability sm...
2,"[finds, reaching, request, clarification, bill...","[find, reach, request, clarification, bill, pa...",find reach request clarification bill payment ...
3,"[reaches, reaching, ask, compatibility, produc...","[reach, ask, compatibility, product, specific,...",reach ask compatibility product specific need ...
4,"[reaches, good, health, eager, learn, features...","[reach, good, health, eager, learn, feature, p...",reach good health eager learn feature product ...


## Save results

In [8]:
df.to_csv('processed_tickets_final_v2.csv', index=False)
print(f'Saved {len(df)} rows to processed_tickets_final_v2.csv')

Saved 28149 rows to processed_tickets_final_v2.csv
